# `lg_workflow_agent` — Simple Checks & Testing

This notebook performs incremental sanity checks on the workflow agent at `src/lg_workflow_agent`:

1. Environment & imports
2. Tool checks (`fetch_trends`, URL validator)
3. Prompt / state shape checks
4. Classifier node in isolation
5. Task generator node in isolation
6. Aggregator node with stubbed sub-agent outputs
7. Validator node with broken-link rewrite path (no LLM)
8. Full end-to-end run (sync)
9. Streaming run with per-step trace

Run cells top-to-bottom. Cells 4-6 and 8-9 require `GOOGLE_API_KEY`.

In [ ]:
# 1. Environment & imports
import os
import sys
from pathlib import Path

# Make `src` importable when running the notebook from tests/
BACKEND_ROOT = Path.cwd()
if BACKEND_ROOT.name == "tests":
    BACKEND_ROOT = BACKEND_ROOT.parent
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from dotenv import load_dotenv
load_dotenv(BACKEND_ROOT / ".env")

print("BACKEND_ROOT:", BACKEND_ROOT)
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("DEEP_AGENT_MODEL:", os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash (default)"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))

## 2. Tool checks

`fetch_trends` (HTTP) and the URL validator (`validate_url`) used by the Validator node.

In [ ]:
from src.lg_workflow_agent.tools import (
    fetch_trends,
    validate_url,
    validate_urls,
    extract_urls,
)

# fetch_trends is a LangChain @tool — invoke via .invoke({...})
sample = fetch_trends.invoke({"source": "hackernews", "topic": "langgraph", "limit": 3, "period": "week"})
print("fetch_trends preview:", (sample or "")[:300], "...\n")

# URL helpers
text = "See https://www.python.org and https://this-domain-should-not-exist-xyz123.invalid for refs."
urls = extract_urls(text)
print("extracted:", urls)
print("validation:", validate_urls(urls))

## 3. Prompt / state shape checks

Quick sanity that prompts format correctly and the state TypedDict carries expected keys.

In [ ]:
from src.lg_workflow_agent import prompts as P
from src.lg_workflow_agent.state import WorkflowState
from src.lg_workflow_agent.nodes import ROLES_BY_TYPE, ROLE_PROMPTS

# Required state keys
required_keys = {
    "query", "task_id", "query_type", "subtasks", "worker_payloads",
    "worker_outputs", "aggregated", "draft_report", "final_report",
    "validation_feedback", "invalid_references", "rewrite_iterations",
}
missing = required_keys - set(WorkflowState.__annotations__.keys())
assert not missing, f"State missing keys: {missing}"
print("State keys OK")

# Roles per type
for qt in ["blog", "comparative", "deep_research", "summary"]:
    roles = ROLES_BY_TYPE[qt]
    for r in roles:
        assert r in ROLE_PROMPTS, f"missing prompt for role {r}"
    print(f"  {qt:14s} -> {roles}")

# Prompt formatting smoke
print(P.CLASSIFIER_PROMPT.format(query="hello")[:120])
print(P.TASK_GENERATOR_PROMPT.format(query="x", query_type="blog", roles="- web_research")[:120])

## 4. Classifier node (isolated)

Initialize an LLM and run only the classifier on a few sample queries.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from src.lg_workflow_agent.nodes import (
    create_node_classifier,
    create_node_task_generator,
    create_node_aggregator,
    create_node_validator,
)

model_name = os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash")
if model_name.startswith("google_genai:"):
    model_name = model_name.split(":", 1)[1]

llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)

# db=None: skip persistence to focus on logic
classify = create_node_classifier(llm, db=None)

samples = [
    "Compare LangGraph vs CrewAI vs AutoGen for multi-agent orchestration",
    "Write a blog post about async Python in 2026",
    "Give me a quick summary of vector databases",
    "Conduct a deep research investigation on quantization-aware training benchmarks",
]
for q in samples:
    out = classify({"query": q, "task_id": "demo"})
    print(f"{q[:55]:55s} -> {out['query_type']:14s} | {out['classification_rationale']}")

## 5. Task generator node (isolated)

Verify the task generator decomposes a query and assigns roles allowed by the query type.

In [ ]:
task_gen = create_node_task_generator(llm, db=None)

state = {
    "query": "Compare LangGraph vs CrewAI for multi-agent orchestration",
    "query_type": "comparative",
    "task_id": "demo",
}
result = task_gen(state)

print(f"Generated {len(result['subtasks'])} subtasks:")
for st in result["subtasks"]:
    print(f"  [{st['id']}] role={st['role']:18s} task={st['task']}")
print("\nWorker payloads (first):")
print(result["worker_payloads"][0])

# Assert role constraints
allowed = set(ROLES_BY_TYPE["comparative"])
assert all(st["role"] in allowed for st in result["subtasks"]), "Role outside allowed set"
print("\nRole assignment OK")

## 6. Aggregator with stubbed sub-agent outputs

Skip real sub-agent calls and feed pre-baked outputs to confirm the aggregator
returns a structured `{metadata, sections, references}` object.

In [ ]:
import json

aggregate = create_node_aggregator(llm, db=None)

stub_outputs = [
    {
        "subtask_id": "s1",
        "role": "web_research",
        "task": "Research LangGraph orchestration model",
        "output": (
            "## Findings\n"
            "LangGraph models workflows as a stateful graph with explicit nodes and edges [1].\n"
            "It supports streaming and human-in-the-loop checkpoints [2].\n"
            "## Sources\n"
            "[1] LangGraph Docs - https://langchain-ai.github.io/langgraph/\n"
            "[2] LangGraph blog - https://blog.langchain.dev/langgraph/"
        ),
    },
    {
        "subtask_id": "s2",
        "role": "content_drafting",
        "task": "Research CrewAI orchestration model",
        "output": (
            "## Draft\n"
            "CrewAI uses a role-based crew abstraction with sequential or hierarchical processes [1].\n"
            "## Sources\n"
            "[1] CrewAI Docs - https://docs.crewai.com/"
        ),
    },
]

state = {
    "query": "Compare LangGraph vs CrewAI",
    "query_type": "comparative",
    "task_id": "demo",
    "worker_outputs": stub_outputs,
}
agg = aggregate(state)["aggregated"]
print(json.dumps(agg, indent=2)[:1500])

## 7. Validator (no LLM)

The validator extracts URLs from the draft, verifies them, and signals a rewrite when broken refs are found.

In [ ]:
validator = create_node_validator(db=None)

draft_good = "Read more at https://www.python.org for details."
draft_bad = (
    "Reference [1] points to https://www.python.org but [2] is broken: "
    "https://this-domain-should-not-exist-xyz123.invalid"
)

print("good draft:")
print(validator({"draft_report": draft_good, "rewrite_iterations": 0}))

print("\nbad draft (1st pass -> rewrite):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 0}))

print("\nbad draft (after MAX_REWRITES -> forced finish):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 2}))

## 8. End-to-end (sync)

Build a fresh `WorkflowAgent` and run a single query through the full graph.

In [ ]:
from src.lg_workflow_agent import WorkflowAgent

agent = WorkflowAgent()
agent.build()
print("Agent ready:", agent.is_ready)

QUERY = "Give a short summary of LangGraph for multi-agent orchestration"
report = agent.invoke(QUERY)
print("\n--- FINAL REPORT ---\n")
print(report)

## 9. Streaming run with per-step trace

Use `astream` to observe how state evolves across nodes (classifier, task_generator, sub-agents, aggregator, writer, validator, cleanup).

In [ ]:
STREAM_QUERY = "Compare LangGraph vs CrewAI for multi-agent orchestration"

trace = []
final_report = ""
async for event in agent.astream(STREAM_QUERY):
    step = event.get("step")
    data = event.get("data", {})
    keys = sorted(k for k in data.keys() if k != "messages")
    line = f"[{step}] keys={keys}"
    if data.get("query_type"):
        line += f" | query_type={data['query_type']}"
    if data.get("subtasks"):
        line += f" | subtasks={len(data['subtasks'])}"
    if data.get("validation_feedback"):
        line += f" | validation={data['validation_feedback']}"
    print(line)
    trace.append(line)
    if data.get("final_report"):
        final_report = data["final_report"]
    elif data.get("draft_report"):
        final_report = data["draft_report"]

print(f"\nTotal events: {len(trace)}")
print("\n--- FINAL REPORT ---\n")
print(final_report or "(no report)")